# Classifier output check

In this notebook I will check the output of the classifier from voting, scoring, sum of weights, confidence, and prediction. 

## Loading & Exploring the data

In [88]:
# Importing the libs
from pathlib import Path
from pprint import pprint

import numpy as np
import plotly.express as px

from kinlens.classifier import Classifier
from kinlens.utils.dataset import DatasetBuilder

In [89]:
# This dataset is a testing dataset (not real).
dataset = DatasetBuilder(Path("../samples/sample-larger.csv"), features=['feature_1', 'feature_2'], label='target').build()

In [90]:
scatter = px.scatter(
    x=dataset.x[:, 0],
    y=dataset.x[:, 1],
    color=dataset.y.astype(str),
    symbol=dataset.y.astype(str),
    labels={
        "x": dataset.features[0],
        "y": dataset.features[1],
        "color": "target",
    },
    title=f"{dataset.name} dataset",
    template="plotly_white",
    opacity=0.8,
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={
        "color": [str(i) for i in range(len(dataset.classes))],
        "symbol": [str(i) for i in range(len(dataset.classes))],
    },
)


scatter.show()

As the figure showing above, we have $3$ classes in the dataset.

In [91]:
class_counts = np.bincount(dataset.y, minlength=len(dataset.classes))
fig = px.bar(
    x=dataset.classes,
    y=class_counts,
    labels={"x": "Class", "y": "Count"},
    title="Class distribution",
    color=dataset.classes,
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.show()
for class_name, count in zip(dataset.classes, class_counts):
    print(f"{class_name}: {count}, percentage: {(count / len(dataset.y)) * 100:.2f}%")

class_0: 58, percentage: 38.67%
class_1: 46, percentage: 30.67%
class_2: 46, percentage: 30.67%


Great, the dataset is almost balanced having both classes `class_1` & `class_2` take the same percent of samples $30.67\%$. And the class `class_0` is the largest class; takes $38.67\%$ percent of the dataset. 

In [92]:
# Defining the KNN classifier with k=5
classifier = Classifier(k=5)

# Training the classifier on the testing dataset.
classifier.fit(dataset=dataset)

# Predicting whitch class does the point [5.0, -1.0] belongs.
explanation = classifier.explain(query=np.array([5.0, -1.0]))
pprint(explanation)

Explanation(prediction=1,
            indices=array([115,  78,  62, 146,  50]),
            labels=array([1, 1, 0, 0, 1]),
            distances=array([2.30022825, 2.57625232, 2.59470711, 2.64556421, 2.7560176 ]),
            weights=array([0.43473947, 0.38816074, 0.38539995, 0.3779912 , 0.36284239]),
            classScore={0: 0.7633911566281458, 1: 1.1857425889974156, 2: 0.0},
            confidence=0.6083433687701402,
            query=array([ 5., -1.]))


$$
\begin{aligned}
\text{Explanation} = \Big(
&\text{prediction}=1,\;
\text{indices}=[115, 78, 62, 146, 50], \\
&\text{labels}=[1,1,0,0,1], \\
&\text{distances}=[2.30022825,\;2.57625232,\;2.59470711,\;2.64556421,\;2.75601760], \\
&\text{weights}=[0.43473947,\;0.38816074,\;0.38539995,\;0.37799120,\;0.36284239], \\
&\text{classScore}=\{0:0.76339116,\;1:1.18574259,\;2:0.0\}, \\
&\text{confidence}=0.60834337,\;
\text{query}=[5,\,-1]
\Big).
\end{aligned}
$$

This means the classifier predicts class $1$, the nearest neighbors are indexed by $[115, 78, 62, 146, 50]$, and the corresponding neighbor labels are $[1,1,0,0,1]$. The class scores are:
$$
\text{score}(0)=0.76339116,\quad
\text{score}(1)=1.18574259,\quad
\text{score}(2)=0.0,
$$
with confidence
$$
0.60834337.
$$

In [93]:
# Extract nearist k neighbors.
neighbors = dataset.x[explanation.indices]
neighbors

array([[6.743, 0.501],
       [6.324, 1.21 ],
       [3.309, 0.968],
       [3.961, 1.433],
       [6.268, 1.447]])

In [94]:
# Extracting the classes of the k nearist neighbors.
classes = dataset.y[explanation.indices]
classes

array([1, 1, 0, 0, 1])

In [95]:
query = px.scatter(
    x=[explanation.query[0]],
    y=[explanation.query[1]],
    color_discrete_sequence=["#d62728"],
    symbol_sequence=["x"],
    labels={
        "x": dataset.features[0],
        "y": dataset.features[1],
    },
    title="Nearest neighbors",
    template="plotly_white",
).data[0]

query.name = "query"
query.marker.size = 14
query.marker.symbol = "x"
query.legendgroup = "query"

fig = px.scatter(
    x=neighbors[:, 0],
    y=neighbors[:, 1],
    color=classes.astype(str),
    symbol=classes.astype(str),
    labels={
        "x": dataset.features[0],
        "y": dataset.features[1],
        "color": "target",
        "symbol": "target",
    },
    title="Nearest neighbors",
    template="plotly_white",
    opacity=0.8,
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={
        "color": [str(i) for i in range(len(dataset.classes))],
        "symbol": [str(i) for i in range(len(dataset.classes))],
    },
)
fig.add_trace(query)

fig.show()

In [96]:
query = px.scatter(
    x=[explanation.query[0]],
    y=[explanation.query[1]],
    color_discrete_sequence=["#d62728"],
    symbol_sequence=["x"],
    labels={
        "x": dataset.features[0],
        "y": dataset.features[1],
    },
    title="Nearest neighbors",
    template="plotly_white",
).data[0]

query.name = "query"
query.marker.size = 14
query.marker.symbol = "x"
query.legendgroup = "query"

fig = px.scatter(
    x=dataset.x[:, 0],
    y=dataset.x[:, 1],
    color=dataset.y.astype(str),
    labels={
        "x": dataset.features[0],
        "y": dataset.features[1],
        "color": "target",
    },
    title="Nearest neighbors",
    template="plotly_white",
    opacity=0.8,
    color_discrete_sequence=px.colors.qualitative.Set2,
    category_orders={
        "color": [str(i) for i in range(len(dataset.classes))],
    },
)

# Outline the K nearest neighbors so they stand out from the dataset.
neighbors_trace = px.scatter(
    x=neighbors[:, 0],
    y=neighbors[:, 1],
).data[0]
neighbors_trace.name = "K nearest neighbors"
neighbors_trace.showlegend = True
neighbors_trace.marker.update(
    symbol="circle-open",
    size=18,
    color="black",
    line={"color": "black", "width": 2},
)
fig.add_trace(neighbors_trace)
fig.add_trace(query)

fig.show()

In [115]:
import numpy


distances = [
    float(numpy.linalg.norm(neighbor - explanation.query))
    for neighbor in dataset.x
    ]

distances: numpy.ndarray = numpy.array(distances, dtype=numpy.float64)

distances

array([ 8.5678017 ,  4.21866448,  6.28564142, 10.54326249,  6.74737712,
        2.8682268 ,  3.81389735,  5.79616856, 10.37886025,  5.10179312,
        3.82324679,  2.92707943,  3.84867003,  9.64203879,  2.82260111,
        4.04861581, 10.02608059,  3.69727089,  2.96595145,  6.01941392,
        5.15757501,  5.54883645,  3.41674187, 10.40707937,  4.49630293,
        3.87228628,  7.37759426,  8.43917816, 11.2003829 ,  9.3417918 ,
        4.87483333,  4.59101655,  8.84022517,  4.59567732,  5.27346234,
        9.24224286,  7.68236845,  3.71656293,  5.42053005,  4.53224062,
        6.06563583, 10.75640948,  6.5827612 ,  9.47695246,  5.98563898,
        6.77828843,  3.5229059 ,  4.02410785,  9.0513151 ,  5.56465857,
        2.7560176 ,  6.31150885,  4.80380734,  6.53094365,  4.60861281,
        3.21217714,  3.96208594,  2.94832308,  3.70883755,  9.11933972,
        3.59365677, 10.27223077,  2.59470711,  9.20602672,  7.52954428,
        5.16164412,  5.92448698,  5.23726532, 10.86876194,  6.18

The nearist K neighbors are identified by the default metric `Euclidean` with the compination with the `BruteForceSearch` searching strategey.

$$
{\delta}_2(x, q) = {|| x - q ||}_2 = \sqrt{\sum_{j=1}^d{(x_j - q_j)^2}}
$$

For a dataset $X=\{x_1,\dots,x_n\}$ and a query $q \in \mathbb{R}^d$, the `BruteForceSearch` strategy evaluates every point in the dataset and computes its Euclidean distance:

$$
d_i = \delta_2(x_i, q) = \|x_i - q\|_2 = \sqrt{\sum_{j=1}^{d}(x_{ij} - q_j)^2}, \quad i=1,\dots,n.
$$

Then it sorts all distances and returns the indices of the $k$ nearest neighbors:

$$
\operatorname{BruteForceSearch}(X, q, k)
=
\operatorname{argsort}(d_1, d_2, \dots, d_n)[:k]
=
\{i_{(1)}, i_{(2)}, \dots, i_{(k)}\},
$$

where

$$
d_{i_{(1)}} \le d_{i_{(2)}} \le \cdots \le d_{i_{(k)}} \le d_j,
\quad
\forall j \notin \{i_{(1)}, i_{(2)}, \dots, i_{(k)}\}.
$$

So the selected samples are exactly the $k$ closest points to $q$ under the Euclidean metric.

In [118]:
indices = numpy.argpartition(distances, 5 - 1)[:5]

indices

array([115,  78,  62, 146,  50])

Let

$$
\mathbf{d} = \left(\delta_2(x_1, q), \delta_2(x_2, q), \ldots, \delta_2(x_n, q)\right),
\quad
\delta_2(x_i, q) = \|x_i - q\|_2.
$$

In this notebook, the code

$$
\text{indices} = \operatorname{np.argpartition}(\mathbf{d},\, k-1)[:k]
$$

with $k=5$ returns the indices of the $k$ smallest distances, without sorting the selected subset internally. Formally, if

$$
I = \{i_1, i_2, \ldots, i_k\}
$$

is the returned index set, then

$$
\forall i \in I,\ \forall j \notin I,\quad d_i \le d_j.
$$

This means the selected samples are the nearest neighbors of the query $q$:

$$
\{x_{i_1}, x_{i_2}, \ldots, x_{i_k}\}
=
\text{KNN}(q, k).
$$

The distances of the selected neighbors are approximately the $k$ smallest values in $\mathbf{d}$, up to the ordering returned by the partition:

$$
(d_{i_1}, d_{i_2}, \ldots, d_{i_k})
\approx
(d_{(1)}, d_{(2)}, \ldots, d_{(k)}),
$$

where $d_{(1)} \le d_{(2)} \le \cdots \le d_{(n)}$ are the ordered distances.

In [122]:
all(indices == explanation.indices)

True

In [126]:
labels = dataset.y[indices]
labels

array([1, 1, 0, 0, 1])

In [127]:
all(labels == explanation.labels)

True

In [157]:
import numpy

neighbors = dataset.x[indices]

distances = numpy.array(
    [float(numpy.linalg.norm(neighbor - explanation.query)) for neighbor in neighbors],
    dtype=numpy.float64,
)

distances: numpy.ndarray = numpy.array(distances, dtype=numpy.float64)

distances

array([2.30022825, 2.57625232, 2.59470711, 2.64556421, 2.7560176 ])

The nearest-neighbor explanation can be written as

$$
I = \{115, 78, 62, 146, 50\},
\quad
\mathbf{y}_I = [1, 1, 0, 0, 1],
\mathbf{d}_I = [2.30022825,\; 2.57625232,\; 2.59470711,\; 2.64556421,\; 2.75601760].
$$

Using the inverse-distance weighting rule with $\epsilon = 10^{-12}$,

$$
w_i = \frac{1}{d_i + \epsilon},
\quad
\mathbf{w} = [0.43473947,\; 0.38816074,\; 0.38539995,\; 0.37799120,\; 0.36284239].
$$

The class scores are computed by summing the weights of neighbors belonging to each class:

$$
s_c = \sum_{i:\, y_i = c} w_i.
$$

Thus,

$$
\mathbf{s} = \{s_0, s_1, s_2\}
= \{0.76339116,\; 1.18574259,\; 0.0\}.
$$

The predicted class is

$$
\hat{y} = \arg\max_c s_c = 1.
$$

The confidence is defined as the ratio between the winning class score and the total score:

$$
\text{confidence}
= \frac{\max_c s_c}{\sum_c s_c}
= \frac{1.18574259}{1.94913375}
\approx 0.60834337.
$$

In [108]:

explanation.distances

array([2.30022825, 2.57625232, 2.59470711, 2.64556421, 2.7560176 ])

In [109]:
all(distances == explanation.distances)

True

In [110]:
explanation.weights

array([0.43473947, 0.38816074, 0.38539995, 0.3779912 , 0.36284239])

In [128]:
power, epsilon = 1, 1e-12

adjusted = distances + epsilon
weights = 1.0 / numpy.power(adjusted, power)

weights

array([0.43473947, 0.38816074, 0.38539995, 0.3779912 , 0.36284239])

In [113]:
all(weights == explanation.weights)

True

In [114]:
explanation

Explanation(prediction=1, indices=array([115,  78,  62, 146,  50]), labels=array([1, 1, 0, 0, 1]), distances=array([2.30022825, 2.57625232, 2.59470711, 2.64556421, 2.7560176 ]), weights=array([0.43473947, 0.38816074, 0.38539995, 0.3779912 , 0.36284239]), classScore={0: 0.7633911566281458, 1: 1.1857425889974156, 2: 0.0}, confidence=0.6083433687701402, query=array([ 5., -1.]))

In [131]:
scores = {code: 0.0 for code in range(len(dataset.classes))}

for label, weight in zip(labels, weights):
    scores[int(label)] += float(weight)

scores

{0: 0.7633911566281458, 1: 1.1857425889974156, 2: 0.0}

In [136]:
scores == explanation.classScore

True

In [149]:
highestScore = float(np.max(list(scores.values())))

prediction = int([label for lable, score in scores.items() if score == highestScore][0])

prediction

1

In [150]:
prediction == explanation.prediction

True

In [154]:
totalScores = numpy.sum(list(scores.values()))

confidence = float(highestScore / totalScores)

confidence

0.6083433687701402

In [155]:
confidence == explanation.confidence

True